# Aramark–Avendra Spend Analysis (EDA)
**CS 562 — Big Data Algorithms | Rutgers University**

## Objective
Explore the Aramark–Avendra dataset to identify spending patterns across customers, industries, locations, and product categories.

- Understand spending distribution
- Identify key drivers of spend
- Explore geographic and business differences
- Generate insights for future modeling

---
⚠️ **Run cells in order from top to bottom. Do not skip any cells.**

## Step 1: Authenticate with Google Cloud
Run this cell first. It will open a popup — sign in with your **@scarletmail.rutgers.edu** account.

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('✅ Authentication complete!')

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

print(f'✅ pandas  {pd.__version__}')
print(f'✅ numpy   {np.__version__}')

## Step 3: Load Data

The file is **~8.5 GB**. We read it in chunks of 100,000 rows at a time so we never run out of memory.

⏱️ **This will take 10–20 minutes. Do not interrupt it.**

In [ ]:
GCS_PATH   = 'gs://cs-562-aramark-project/Andrew_Meszaros_SRF_2026-04-01-0936.csv'
CHUNK_SIZE = 100_000
SPEND_COL  = 'Spend Random Factor'

chunks = []
total  = 0

print('Starting chunked read...')
for i, chunk in enumerate(pd.read_csv(GCS_PATH, chunksize=CHUNK_SIZE, dtype={'Zip': str})):
    chunks.append(chunk)
    total += len(chunk)
    if (i + 1) % 10 == 0:
        print(f'  ...loaded {total:,} rows so far')

df = pd.concat(chunks, ignore_index=True)
print(f'\n✅ Data loaded! Shape: {df.shape}')
display(df.head())

## Step 4: Dataset Overview

In [ ]:
print(f'Rows:    {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'\nColumn names:')
for col in df.columns:
    print(f'  - {col}')

In [ ]:
df.info()

In [ ]:
df.describe()

## Step 5: Data Cleaning

In [ ]:
print('Missing values per column:')
display(df.isnull().sum().sort_values(ascending=False))

In [ ]:
before = len(df)
df = df.dropna(subset=[SPEND_COL])
df[SPEND_COL] = pd.to_numeric(df[SPEND_COL], errors='coerce')
df = df.dropna(subset=[SPEND_COL])
after = len(df)

print(f'Rows before cleaning: {before:,}')
print(f'Rows after  cleaning: {after:,}')
print(f'Rows dropped:         {before - after:,}')

## Step 6: Exploratory Analysis

### 6.1 Spend by Category Level 1
Which top-level product categories drive the most total spend?

In [ ]:
category_spend = df.groupby('Category Level 1')[SPEND_COL].sum().sort_values(ascending=False)
display(category_spend.reset_index().rename(columns={SPEND_COL: 'Total Spend (USD)'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
category_spend.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Total Spend by Category Level 1', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Spend (USD)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.2 Spend by State
Geographic distribution of spend across US states.

In [ ]:
state_spend = df.groupby('State')[SPEND_COL].sum().sort_values(ascending=False)
display(state_spend.head(10).reset_index().rename(columns={SPEND_COL: 'Total Spend (USD)'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
state_spend.head(10).plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 10 States by Total Spend', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Spend (USD)')
ax.set_xlabel('State')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.3 Spend by Business Entity Type
Average spend per transaction across different business types.

In [ ]:
business_spend = df.groupby('Business Entity Type')[SPEND_COL].mean().sort_values(ascending=False)
display(business_spend.reset_index().rename(columns={SPEND_COL: 'Avg Spend (USD)'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
business_spend.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_title('Average Spend by Business Entity Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Spend (USD)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.4 Spend by City
Top 10 cities by total spend.

In [ ]:
city_spend = df.groupby('City')[SPEND_COL].sum().sort_values(ascending=False)
display(city_spend.head(10).reset_index().rename(columns={SPEND_COL: 'Total Spend (USD)'}))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
city_spend.head(10).plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Top 10 Cities by Total Spend', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Spend (USD)')
ax.set_xlabel('City')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 6.5 Ecommerce vs Non-Ecommerce Spending
Does ecommerce adoption correlate with higher average spend?

In [ ]:
ecom_spend = df.groupby('Ecommerce Status')[SPEND_COL].mean().sort_values(ascending=False)
display(ecom_spend.reset_index().rename(columns={SPEND_COL: 'Avg Spend (USD)'}))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ecom_spend.plot(kind='bar', ax=ax, color=['steelblue', 'lightgray'], edgecolor='white')
ax.set_title('Average Spend: Ecommerce vs Non-Ecommerce', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Spend (USD)')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 6.6 Spend Distribution
What does the overall distribution of spend look like?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df[SPEND_COL].dropna(), bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Spend Distribution (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Spend (USD)')
axes[0].set_ylabel('Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Log distribution (handles skew better)
log_spend = np.log1p(df[SPEND_COL].dropna())
axes[1].hist(log_spend, bins=100, color='coral', edgecolor='white')
axes[1].set_title('Spend Distribution (Log Scale)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Spend + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 6.7 Top Category Level 2 within each Category Level 1
Drilling one level deeper into the category hierarchy.

In [ ]:
cat2_spend = (
    df.groupby(['Category Level 1', 'Category Level 2'])[SPEND_COL]
    .sum()
    .reset_index()
    .sort_values(SPEND_COL, ascending=False)
)

# Top 3 sub-categories per Category Level 1
top_cat2 = cat2_spend.groupby('Category Level 1').head(3)
display(top_cat2.rename(columns={SPEND_COL: 'Total Spend (USD)'}))

## Step 7: Key Insights

Fill this in after running the analysis above:

- **Top category:** _[fill in after running]_
- **Top state:** _[fill in after running]_
- **Top city:** _[fill in after running]_
- **Business type with highest avg spend:** _[fill in after running]_
- **Ecommerce impact:** _[fill in after running]_

These insights provide the foundation for next steps: clustering, spend prediction, and association rule mining.

## Step 8: Save Results

In [ ]:
os.makedirs('results', exist_ok=True)

category_spend.to_csv('results/category_spend.csv')
state_spend.to_csv('results/state_spend.csv')
city_spend.to_csv('results/city_spend.csv')
business_spend.to_csv('results/business_spend.csv')
ecom_spend.to_csv('results/ecom_spend.csv')

print('✅ Results saved to /results folder!')
print('   To download: click the folder icon on the left sidebar in Colab')

## Next Steps

Based on this EDA, the project will move toward:

1. **Customer Segmentation** — K-Means or DBSCAN clustering on spend patterns by location + category
2. **Spend Prediction** — Regression / ML models using business type, location, category, ecommerce status
3. **Association Rule Mining** — Frequent itemsets across categories per customer (MMDS Chapter 6)

Final direction will be selected based on data patterns observed in this EDA.